# EEG Bandpass Filtering Demo

Inspect the same real e38 Stroop recording with the validated reusable EEG helpers used by the main notebook. The helpers verify the recording and requested time window, exclude mastoid/EOG/Status channels, and check finite raw and filtered values.

This corrected demonstration uses the project’s zero-phase FIR 0.1–20 Hz filter on 61 scalp channels. It replaces the old notebook-local Butterworth implementation and synthetic-data fallback; no study statistic depends on this demonstration.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display

REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / 'code' / 'src' / 'posthoc_analysis').is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Open this notebook from the repository or notebooks directory.')
sys.path.insert(0, str(REPO_ROOT / 'code' / 'src'))
from posthoc_analysis.config import PROJECT_ROOT
from posthoc_analysis.eeg import (
    load_and_plot_eeg_run_segment,
    load_filter_and_plot_eeg_run_segment,
)


## Recording and raw EEG

Use the original demonstration recording and the 30–33 s segment. A missing recording raises an informative error. Both plots use the same 61 recorded scalp electrodes.


In [ ]:
gdf_path = (PROJECT_ROOT / 'e38' / 'e38_20260228' /
            'e38_20260228162653_stroop' / 'e38_20260228162653_stroop.gdf')
raw_eeg_demo = load_and_plot_eeg_run_segment(
    gdf_path=gdf_path, task='stroop', start_sec=30.0, duration_sec=3.0,
)
print('Validated raw segment:', raw_eeg_demo['segment_samples_by_channels'].shape)
display(raw_eeg_demo['fig'])


## Zero-phase filtering

Filter the full continuous recording before selecting the display segment. The spectrum provides a separate inspection of the filter’s effect; the traces remain descriptive QC.


In [ ]:
filtered_eeg_demo = load_filter_and_plot_eeg_run_segment(
    gdf_path=gdf_path, task='stroop', l_freq=0.1, h_freq=20.0,
    start_sec=30.0, duration_sec=3.0,
)
assert filtered_eeg_demo['eeg_labels'] == raw_eeg_demo['eeg_labels']
assert filtered_eeg_demo['filtered_segment_samples_by_channels'].shape == raw_eeg_demo['segment_samples_by_channels'].shape
print('Validated filtered segment:', filtered_eeg_demo['filtered_segment_samples_by_channels'].shape)
display(filtered_eeg_demo['trace_fig'])


In [ ]:
display(filtered_eeg_demo['psd_fig'])


## Interpretation

The raw and filtered views use the same real samples and scalp channels. Filtering attenuates activity outside the selected passband; this visual inspection is not a behavioral or physiological hypothesis test. All loading, validation, filtering, and plotting logic lives in `code/src/posthoc_analysis/eeg.py`.
